In [1]:
# ============================================================
# INGEST CONSTRUCTORS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql.types import StructType, StructField, StringType
import nbformat
from nbconvert import PythonExporter

In [2]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 13:17:51 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/09 13:17:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 13:17:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 13:17:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/09 13:17:51 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/09 13:17:51 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [3]:
# -----------------------------------------
# 2. Detect latest landing batch folder
# -----------------------------------------
batch_folders = sorted([
    f for f in os.listdir(LANDING_PATH)
    if os.path.isdir(os.path.join(LANDING_PATH, f))
])

if not batch_folders:
    raise Exception("❌ No batch folders found in landing!")

p_batch_id = batch_folders[-1]
print("Detected landing batch folder:", p_batch_id)

BATCH_LANDING_PATH = f"{LANDING_PATH}/{p_batch_id}"

Detected landing batch folder: 2025-01


In [4]:
# -----------------------------------------
# 3. Generate pipeline batch_id
# -----------------------------------------
batch_id = generate_batch_id()
print("Pipeline batch_id:", batch_id)

Pipeline batch_id: 20260609_131752


In [5]:
# -----------------------------------------
# 4. Define source + target paths
# -----------------------------------------
source_file = f"{BATCH_LANDING_PATH}/constructors.json"
target_path = f"{BRONZE_PATH}/constructors"
os.makedirs(target_path, exist_ok=True)

print("Source file:", source_file)
print("Target path:", target_path)

Source file: /Users/manoharazalki/F1-Analytics/data/landing/2025-01/constructors.json
Target path: /Users/manoharazalki/F1-Analytics/data/bronze/constructors


In [6]:
# -----------------------------------------
# 5. Define schema
# -----------------------------------------
constructors_schema = StructType([
    StructField("constructorId", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

In [7]:
# -----------------------------------------
# 6. Read JSON
# -----------------------------------------
constructors_df = (
    spark.read
        .format("json")
        .schema(constructors_schema)
        .option("mode", "FAILFAST")
        .load(source_file)
)

print("✔ Constructors file read successfully")
constructors_df.show(5, truncate=False)


✔ Constructors file read successfully
+-------------+----------+-----------+-----------------------------------------------------------------+
|constructorId|name      |nationality|url                                                              |
+-------------+----------+-----------+-----------------------------------------------------------------+
|adams        |Adams     |american   |http://en.wikipedia.org/wiki/Adams_(constructor)                 |
|afm          |AFM       |german     |http://en.wikipedia.org/wiki/Alex_von_Falkenhausen_Motorenbau    |
|ags          |AGS       |french     |http://en.wikipedia.org/wiki/Automobiles_Gonfaronnaises_Sportives|
|alfa         |Alfa Romeo|swiss      |http://en.wikipedia.org/wiki/Alfa_Romeo_in_Formula_One           |
|alphatauri   |AlphaTauri|italian    |http://en.wikipedia.org/wiki/Scuderia_AlphaTauri                 |
+-------------+----------+-----------+-----------------------------------------------------------------+
only showing top 

In [8]:
# -----------------------------------------
# 7. Add ingestion metadata
# -----------------------------------------
constructors_final_df = add_ingestion_metadata(constructors_df, source_file)

print("✔ Metadata added")
constructors_final_df.show(5, truncate=False)

✔ Metadata added
+-------------+----------+-----------+-----------------------------------------------------------------+--------------------------+------------------------------------------------------------------------+
|constructorId|name      |nationality|url                                                              |ingest_timestamp          |source_file                                                             |
+-------------+----------+-----------+-----------------------------------------------------------------+--------------------------+------------------------------------------------------------------------+
|adams        |Adams     |american   |http://en.wikipedia.org/wiki/Adams_(constructor)                 |2026-06-09 13:17:53.801246|/Users/manoharazalki/F1-Analytics/data/landing/2025-01/constructors.json|
|afm          |AFM       |german     |http://en.wikipedia.org/wiki/Alex_von_Falkenhausen_Motorenbau    |2026-06-09 13:17:53.801246|/Users/manoharazalki/F1-Analytic

In [9]:
# -----------------------------------------
# 8. Write to Bronze (partitioned by batch_id)
# -----------------------------------------
write_to_bronze(
    constructors_final_df,
    f"{target_path}/data.parquet",
    batch_id
)

print(f"✔ Constructors Bronze written to: {target_path}/data.parquet")

✔ Constructors Bronze written to: /Users/manoharazalki/F1-Analytics/data/bronze/constructors/data.parquet


In [10]:
# -----------------------------------------
# 9. Read back for validation
# -----------------------------------------
spark.read.parquet(f"{target_path}/data.parquet").show(5, truncate=False)

+-------------+----------+-----------+-----------------------------------------------------------------+--------------------------+------------------------------------------------------------------------+---------------+
|constructorId|name      |nationality|url                                                              |ingest_timestamp          |source_file                                                             |batch_id       |
+-------------+----------+-----------+-----------------------------------------------------------------+--------------------------+------------------------------------------------------------------------+---------------+
|adams        |Adams     |american   |http://en.wikipedia.org/wiki/Adams_(constructor)                 |2026-06-09 13:17:53.880983|/Users/manoharazalki/F1-Analytics/data/landing/2025-01/constructors.json|20260609_131752|
|afm          |AFM       |german     |http://en.wikipedia.org/wiki/Alex_von_Falkenhausen_Motorenbau    |2026-06-09 1